# Decision Tree — Network Intrusion Detection

Huấn luyện và đánh giá mô hình **Decision Tree** trên tập dữ liệu CICIDS2017.
Decision Tree không yêu cầu feature scaling.

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier

# Thêm thư mục gốc dự án vào sys.path
sys.path.insert(0, str(Path.cwd()))
sys.path.append(os.path.abspath(".."))

from src.model_training import (
    load_splits,
    evaluate_model,
    plot_confusion_matrix,
    compare_models,
)

In [ ]:
# --- STEP 1: Load dữ liệu đã chia sẵn ---
print("=" * 70)
print("STEP 1: Loading pre-split data")
print("=" * 70)

X_train, X_test, y_train, y_test = load_splits()

print(f"Feature columns: {X_train.shape[1]}")
print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Classes: {sorted(y_train.unique())}")

In [ ]:
# --- STEP 2: Train Decision Tree ---
print("\n" + "=" * 70)
print("STEP 2: Training Decision Tree (max_depth=20)")
print("=" * 70)

t0 = time.time()

dt_model = DecisionTreeClassifier(
    max_depth=20,
    random_state=42,
)

print("Fitting model on training data (no scaling needed)...")
dt_model.fit(X_train, y_train)

train_time = time.time() - t0
print(f"\nTraining completed in {train_time:.2f} seconds")
print(f"Tree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")

In [ ]:
# --- STEP 3: Evaluate ---
print("\n" + "=" * 70)
print("STEP 3: Model Evaluation")
print("=" * 70)

y_pred = dt_model.predict(X_test)
y_pred_proba = dt_model.predict_proba(X_test)

dt_results = evaluate_model(
    y_true=y_test,
    y_pred=y_pred,
    model_name="Decision Tree (depth=20)",
    y_pred_proba=y_pred_proba,
    labels=dt_model.classes_.tolist(),
)

In [ ]:
# --- STEP 4: Confusion Matrix ---
print("\n" + "=" * 70)
print("STEP 4: Confusion Matrix")
print("=" * 70)

plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred,
    labels=dt_model.classes_.tolist(),
    model_name="Decision Tree (depth=20)",
    normalize=True,
    figsize=(12, 10),
)

In [ ]:
# --- STEP 5: Feature Importance ---
print("\n" + "=" * 70)
print("STEP 5: Feature Importance (Top 20)")
print("=" * 70)

feat_imp = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": dt_model.feature_importances_,
}).sort_values("Importance", ascending=False)

print(feat_imp.head(20).to_string(index=False))

plt.figure(figsize=(12, 6))
sns.barplot(
    data=feat_imp.head(20),
    x="Importance", y="Feature",
    hue="Feature", legend=False,
    palette="viridis",
)
plt.title("Top 20 Feature Importance — Decision Tree")
plt.xlabel("Importance Score")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# --- STEP 6: Summary ---
print("\n" + "=" * 70)
print("STEP 6: Summary")
print("=" * 70)

comparison_df = compare_models([dt_results])
print("\nPerformance Metrics:")
print(comparison_df.to_string(index=False))

print(f"\nTraining time : {train_time:.2f}s")
print(f"Test set size : {len(y_test):,}")
print(f"Features      : {X_train.shape[1]}")
print(f"Classes       : {len(dt_model.classes_)}")